# 02: Prepare the blind coding sheet
One row per record, **no Opus codes, notes or vocabulary**. Rows are shuffled within arm (fixed seed) so stratum order and the original coding order are not reproduced. The 75 records of the planned 10% subset (`kappa_sheet.tsv`) are flagged `priority_10pct = 1`: a coder with limited time does those first; everything else is a bonus. The xlsx has drop-down lists for every coded field. **Never overwrites a sheet that already contains codes.**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 80)

In [ ]:
import hashlib, datetime
from io_utils import FROZEN, CODING, source, load_coder_sheet, settings
from coding_schema import SHEET_COLUMNS, CODE_FIELDS, CODEBOOK
frame = pd.read_csv(FROZEN / "sample_frame.tsv", sep="\t", dtype=str, keep_default_na=False)
prio = set(pd.read_csv(source("kappa_sheet"), sep="\t", dtype=str)["pmid"])
assert len(prio) == 75 and prio <= set(frame.pmid)  # 75 distinct PMIDs
frame["priority_10pct"] = frame.pmid.isin(prio).map({True: "1", False: ""})
seed = settings()["sheet_order_seed"]
frame["_k"] = [hashlib.sha256(f"{seed}-{p}".encode()).hexdigest() for p in frame.pmid]
frame = (frame.sort_values(["arm", "priority_10pct", "_k"], ascending=[True, False, True])
              .drop(columns="_k"))
for c in SHEET_COLUMNS:
    if c not in frame:
        frame[c] = ""
sheet = frame[SHEET_COLUMNS].reset_index(drop=True)
xlsx, tsv = CODING / "coding_sheet.xlsx", CODING / "coding_sheet.tsv"
if xlsx.exists() and (load_coder_sheet(xlsx)[CODE_FIELDS] != "").any().any():
    raise SystemExit("coding_sheet.xlsx already contains codes: not overwritten")
sheet.to_csv(tsv, sep="\t", index=False)
sheet.shape, sheet.priority_10pct.eq("1").sum()

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
wb = Workbook()
ws = wb.active; ws.title = "coding"
ws.append(SHEET_COLUMNS)
for row in sheet.itertuples(index=False):
    ws.append(list(row))
lists = wb.create_sheet("lists")
widths = {"title": 45, "abstract": 90, "notes": 40, "evidence_quote": 40, "meaning": 30, "journal": 20}
code_fill = PatternFill("solid", fgColor="FFF4CC")
for j, col in enumerate(SHEET_COLUMNS, 1):
    L = get_column_letter(j)
    ws.column_dimensions[L].width = widths.get(col, 14)
    ws.cell(1, j).font = Font(bold=True)
    if col in CODE_FIELDS + ["evidence_quote", "notes"]:
        ws.cell(1, j).fill = code_fill
    if col in CODEBOOK:
        k = list(CODEBOOK).index(col) + 1
        LL = get_column_letter(k)
        lists.cell(1, k, col)
        for i, v in enumerate(CODEBOOK[col], 2):
            lists.cell(i, k, v)
        dv = DataValidation(type="list", formula1=f"=lists!${LL}$2:${LL}${len(CODEBOOK[col]) + 1}", allow_blank=True)
        ws.add_data_validation(dv); dv.add(f"{L}2:{L}{len(sheet) + 1}")
for r in range(2, len(sheet) + 2):
    ws.row_dimensions[r].height = 120
    for c in (SHEET_COLUMNS.index("title") + 1, SHEET_COLUMNS.index("abstract") + 1):
        ws.cell(r, c).alignment = Alignment(wrap_text=True, vertical="top")
ws.freeze_panes = "J2"
lists.sheet_state = "hidden"
wb.save(xlsx)
print("wrote", xlsx.name, "and", tsv.name, datetime.date.today())